# Notebook 4 — Memory Management for AI Agents

---

## Why Does Memory Matter?

Imagine having a conversation with someone who forgets everything you said the moment you stop talking. That is the default state of an LLM — each API call is completely stateless. The model has no memory of what you discussed five minutes ago, let alone last week.

Real assistants — human or AI — need **memory** to be useful:

- A doctor remembers that you are allergic to penicillin, even if you visited six months ago.
- A customer service agent remembers you already tried rebooting, so they don't ask again.
- A tutor remembers which topics you struggled with last semester.

This notebook teaches you how to give your AI agent the same capability by building **three distinct memory layers** — each inspired by how human memory actually works.

---

## The Three Memory Layers

```
┌──────────────────────────────────────────────────────────────┐
│                  AI Agent Memory Architecture                 │
│                                                               │
│  ┌─────────────────────┐                                      │
│  │  SHORT-TERM MEMORY  │  ← Last N messages (sliding window)  │
│  │  (Conversation      │    Fits inside the LLM context window│
│  │   Buffer)           │    Oldest messages dropped first     │
│  └─────────────────────┘                                      │
│                                                               │
│  ┌─────────────────────┐                                      │
│  │  EPISODIC MEMORY    │  ← Summaries of past conversations   │
│  │  (FAISS Vector DB)  │    Stored as semantic embeddings     │
│  │                     │    Retrieved by meaning, not keyword │
│  └─────────────────────┘                                      │
│                                                               │
│  ┌─────────────────────┐                                      │
│  │  LONG-TERM MEMORY   │  ← Extracted facts about the user    │
│  │  (FAISS Vector DB)  │    Permanent, semantic recall        │
│  │                     │    Injected into every system prompt │
│  └─────────────────────┘                                      │
│                                                               │
│  All three layers combine into one memory-augmented prompt    │
│  that gets passed to the LLM on every turn.                   │
└──────────────────────────────────────────────────────────────┘
```

| Memory Type | Human Analogy | Storage | How Retrieved |
|-------------|--------------|---------|---------------|
| Short-Term  | Working memory — what's in your head RIGHT NOW | Python list | Always included (sliding window) |
| Episodic    | "I remember our chat last Tuesday about..." | FAISS vector index | Semantic similarity search |
| Long-Term   | "I know your name is Alice and you have a dog" | FAISS vector index | Semantic similarity search |

---

## What You Will Build

By the end of this notebook, you will have a working **memory-augmented chatbot** that:

1. **Truncates conversation history** when it gets too long (Short-Term Memory)
2. **Stores summaries** of old conversations and recalls them when relevant (Episodic Memory)
3. **Extracts facts** about you from conversation and recalls them forever (Long-Term Memory)
4. **Combines all three** into a richer system prompt on every turn

---

## Prerequisites

- Python 3.9+
- An `ANTHROPIC_API_KEY` and an `OPENAI_API_KEY` set in a `.env` file (OpenAI is used for embeddings only)
- `faiss-cpu`, `tiktoken`, `openai`, `anthropic`, and `numpy` installed (see Step 0)

---
## Step 0 — Install & Import Dependencies

We need a few libraries:

- **`faiss-cpu`** — Facebook AI Similarity Search. A blazing-fast library for finding the most similar items in a collection of vectors. Think of it as a *search engine for meaning*.
- **`tiktoken`** — OpenAI's tokenizer. Counts how many tokens a piece of text uses, so we can stay within the LLM's context window limit.
- **`numpy`** — Numerical arrays, needed by FAISS.
- **`openai`** — Used only to generate text embeddings (turning text into a vector of numbers).
- **`anthropic`** — The actual LLM that will chat with us.

In [1]:
import os
import json
import math
from typing import Optional
from dotenv import load_dotenv

import numpy as np
import faiss
import tiktoken
import openai
import anthropic

load_dotenv()

if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = ''
    
openai_client  = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
claude_client  = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

EMBED_MODEL    = "text-embedding-3-small"  # 1 536-dimensional vectors
EMBED_DIM      = 1536
CLAUDE_MODEL   = "claude-sonnet-4-6"

print("✅ Imports complete.")
print(f"   Embedding model : {EMBED_MODEL}  ({EMBED_DIM} dims)")
print(f"   LLM             : {CLAUDE_MODEL}")

✅ Imports complete.
   Embedding model : text-embedding-3-small  (1536 dims)
   LLM             : claude-sonnet-4-6


---
## Step 1 — Turning Text into Vectors (Embeddings)

Before we can store anything in FAISS, we need to convert text into numbers — specifically, a **vector** (a list of floating-point numbers) that captures the *meaning* of the text.

### What is an embedding?

An embedding is like a coordinate in a high-dimensional space. Texts with similar meanings end up close together in this space:

```
  "I love dogs"  ────────────┐
                             │  ← close together (both about pets)
  "My puppy is cute" ────────┘

  "The stock market fell"  ──── far away from the above
```

FAISS lets us ask: *"Given this query, which stored texts are closest in meaning?"*

### Cosine Similarity vs L2 Distance

We use **cosine similarity** (how much do two vectors point in the same direction?). It is better than raw distance for text because it ignores the overall length of sentences and focuses purely on meaning.

To use cosine similarity in FAISS, we **normalise** each vector to unit length first, then use an `IndexFlatIP` (Inner Product) index. Normalised vectors + inner product = cosine similarity.

In [2]:
def embed(text: str) -> np.ndarray:
    """
    Convert a string into a normalised 1536-dimensional float32 vector.
    Normalisation lets us use FAISS IndexFlatIP as cosine similarity.
    """
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=[text])
    vec = np.array(response.data[0].embedding, dtype=np.float32)
    # L2-normalise so ||vec|| = 1  →  inner product == cosine similarity
    vec /= np.linalg.norm(vec)
    return vec


# ── Quick smoke-test ────────────────────────────────────────────────────────
v1 = embed("I love dogs")
v2 = embed("My puppy is so cute")
v3 = embed("The quarterly earnings report exceeded forecasts")

sim_12 = float(np.dot(v1, v2))
sim_13 = float(np.dot(v1, v3))

print(f"Shape of embedding vector : {v1.shape}")
print(f"Cosine similarity ('dogs' vs 'puppy')    : {sim_12:.4f}  ← should be HIGH")
print(f"Cosine similarity ('dogs' vs 'earnings') : {sim_13:.4f}  ← should be LOW")

Shape of embedding vector : (1536,)
Cosine similarity ('dogs' vs 'puppy')    : 0.5150  ← should be HIGH
Cosine similarity ('dogs' vs 'earnings') : 0.1114  ← should be LOW


---
## Step 2 — Short-Term Memory: The Sliding Window Buffer

### The Problem: LLMs Have a Finite Context Window

Every LLM can only process a limited number of tokens in a single call. Claude 3.5 Haiku allows up to 200,000 tokens, but in practice you don't want to send thousands of old messages on every turn — it is expensive and slow.

**Short-Term Memory** solves this with a **sliding window**: keep only the last N tokens of conversation history, and drop the oldest messages when you exceed the limit.

```
Conversation grows over time:

  [msg1] [msg2] [msg3] [msg4] [msg5]  ← window fills up
                                                      ↓ new message arrives
         [msg2] [msg3] [msg4] [msg5] [msg6]  ← msg1 dropped (oldest first)
```

### Tokens vs Characters

A **token** is roughly 4 characters or ¾ of a word. `tiktoken` counts them exactly so we can set a precise limit.

```
"Hello, world!"  →  4 tokens
"The quick brown fox jumps over the lazy dog"  →  10 tokens
```

In [3]:
class ShortTermMemory:
    """
    A token-aware sliding-window conversation buffer.

    Keeps the most recent messages that fit within `max_tokens`.
    When the buffer overflows, the OLDEST messages are dropped first.
    """

    def __init__(self, max_tokens: int = 2000):
        self.max_tokens = max_tokens
        self.messages: list[dict] = []          # [{"role": ..., "content": ...}, ...]
        # cl100k_base is the tokeniser used by GPT-4 and Claude-compatible models
        self._enc = tiktoken.get_encoding("cl100k_base")

    # ── helpers ──────────────────────────────────────────────────────────────

    def _count_tokens(self, text: str) -> int:
        """Return the number of tokens in a string."""
        return len(self._enc.encode(text))

    def _total_tokens(self) -> int:
        """Sum of tokens across all stored messages."""
        return sum(self._count_tokens(m["content"]) for m in self.messages)

    def _truncate_if_needed(self):
        """
        Drop the oldest messages until the buffer fits within max_tokens.
        Always keeps at least one message (the most recent).
        """
        while self._total_tokens() > self.max_tokens and len(self.messages) > 1:
            dropped = self.messages.pop(0)       # remove the oldest
            print(f"   [STM] Dropped oldest message ({self._count_tokens(dropped['content'])} tokens): "
                  f'"{dropped["content"][:50]}..."')

    # ── public API ───────────────────────────────────────────────────────────

    def add(self, role: str, content: str):
        """Add a new message and truncate if necessary."""
        self.messages.append({"role": role, "content": content})
        self._truncate_if_needed()

    def get_messages(self) -> list[dict]:
        """Return the current window as a list of message dicts."""
        return list(self.messages)

    def status(self) -> str:
        total = self._total_tokens()
        return (f"ShortTermMemory: {len(self.messages)} messages, "
                f"{total}/{self.max_tokens} tokens used "
                f"({100*total/self.max_tokens:.1f}%)")


print("✅ ShortTermMemory class defined.")

✅ ShortTermMemory class defined.


### Demo: Watch Short-Term Memory Truncate

Let's force truncation by setting a tiny token limit and adding several messages. You'll see the oldest ones get dropped automatically.

In [4]:
# Use a tiny limit so we can observe truncation happening
stm_demo = ShortTermMemory(max_tokens=80)

conversation_sample = [
    ("user",      "Hi! My name is Alice and I'm studying machine learning."),
    ("assistant", "Hello Alice! That's a fascinating field. What aspect are you focusing on?"),
    ("user",      "I'm interested in natural language processing and transformer models."),
    ("assistant", "NLP is exciting! Transformers have really revolutionised the field."),
    ("user",      "Can you explain attention mechanisms?"),
    ("assistant", "Sure! Attention lets the model focus on relevant parts of the input when generating each token."),
]

print("Adding messages one by one (limit = 80 tokens):\n")
for role, content in conversation_sample:
    print(f"  + [{role}] {content[:60]}")
    stm_demo.add(role, content)

print(f"\n{stm_demo.status()}")
print("\nRemaining messages in window:")
for m in stm_demo.get_messages():
    print(f"  [{m['role']}] {m['content'][:70]}")

Adding messages one by one (limit = 80 tokens):

  + [user] Hi! My name is Alice and I'm studying machine learning.
  + [assistant] Hello Alice! That's a fascinating field. What aspect are you
  + [user] I'm interested in natural language processing and transforme
  + [assistant] NLP is exciting! Transformers have really revolutionised the
  + [user] Can you explain attention mechanisms?
  + [assistant] Sure! Attention lets the model focus on relevant parts of th

ShortTermMemory: 6 messages, 77/80 tokens used (96.2%)

Remaining messages in window:
  [user] Hi! My name is Alice and I'm studying machine learning.
  [assistant] Hello Alice! That's a fascinating field. What aspect are you focusing 
  [user] I'm interested in natural language processing and transformer models.
  [assistant] NLP is exciting! Transformers have really revolutionised the field.
  [user] Can you explain attention mechanisms?
  [assistant] Sure! Attention lets the model focus on relevant parts of the input wh


---
## Step 3 — Episodic Memory: Remembering Past Conversations

### The Problem: Short-Term Memory Forgets

Short-term memory is great for the current conversation, but once a conversation ends, the context is lost. If a user says "remember last week when we talked about Python?" — a pure short-term memory agent has no idea.

**Episodic Memory** stores *summaries* of past conversations as embeddings in a vector database (FAISS). When a new conversation starts, we **search** the episodic store to find past episodes that are *semantically relevant* to what the user is currently asking.

### How FAISS Works

FAISS is a library that can store millions of vectors and find the top-K most similar ones in milliseconds. Think of it as a phonebook, but instead of looking up names alphabetically, you look up *meanings* by similarity.

```
FAISS Index (vector database):

  ID 0 → [0.12, -0.45, 0.89, ...]  ← embedding of "session 1 summary"
  ID 1 → [0.33,  0.71, 0.02, ...]  ← embedding of "session 2 summary"
  ID 2 → [-0.5,  0.11, 0.64, ...]  ← embedding of "session 3 summary"
  ...

Query: embed("Python programming question")
  → FAISS returns [ID 1, ID 0] as the most similar episodes
```

A parallel Python list stores the actual text, keyed by the same integer IDs.

In [5]:
class EpisodicMemory:
    """
    Stores conversation episode summaries as vector embeddings in a FAISS index.
    On each new turn, retrieves the K most semantically relevant past episodes.
    """

    def __init__(self, embed_dim: int = EMBED_DIM):
        # IndexFlatIP = exact inner-product search.
        # With normalised vectors this equals cosine similarity.
        self.index = faiss.IndexFlatIP(embed_dim)
        self.episodes: list[str] = []   # parallel list: episodes[i] ↔ index vector i

    def store_episode(self, summary: str):
        """
        Embed a conversation summary and add it to the FAISS index.
        """
        vec = embed(summary).reshape(1, -1)   # FAISS expects shape (n, dim)
        self.index.add(vec)
        self.episodes.append(summary)
        print(f"   [Episodic] Stored episode #{len(self.episodes)}: {summary[:60]}...")

    def retrieve(self, query: str, k: int = 3) -> list[str]:
        """
        Find the K most relevant past episodes for a given query.
        Returns an empty list if no episodes are stored.
        """
        if self.index.ntotal == 0:
            return []

        k = min(k, self.index.ntotal)          # can't retrieve more than we have
        q_vec = embed(query).reshape(1, -1)
        scores, indices = self.index.search(q_vec, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0:                        # FAISS returns -1 for empty slots
                results.append((float(score), self.episodes[idx]))

        # Sort by score descending (most relevant first)
        results.sort(key=lambda x: x[0], reverse=True)
        return [text for _, text in results]

    def __len__(self):
        return self.index.ntotal


print("✅ EpisodicMemory class defined.")

✅ EpisodicMemory class defined.


### Demo: Store and Retrieve Episodes

Let's simulate having had several past conversations and then ask a question. Watch how FAISS retrieves the most relevant episodes.

In [6]:
ep_mem = EpisodicMemory()

print("Storing past conversation episodes...\n")
past_episodes = [
    "Session on 2024-01-10: User asked about Python list comprehensions and lambda functions. "
    "We practised filtering lists and mapping transformations.",

    "Session on 2024-01-15: User wanted help with a SQL query joining three tables. "
    "We debugged a GROUP BY clause and fixed a WHERE condition.",

    "Session on 2024-01-20: User asked about neural network training. "
    "We discussed gradient descent, loss functions, and overfitting prevention.",

    "Session on 2024-01-25: User was building a Flask web API. "
    "We set up routes, added JSON responses, and handled POST requests.",

    "Session on 2024-02-01: User asked about transformer attention mechanisms. "
    "We walked through self-attention, multi-head attention, and positional encoding.",
]

for ep in past_episodes:
    ep_mem.store_episode(ep)

# ── Now retrieve relevant episodes for a new query ──────────────────────────
print(f"\n{'─'*60}")
query = "Can you help me understand how transformers process language?"
print(f"\nUser query: \"{query}\"")
print("\nTop 2 retrieved episodes:")

for i, ep in enumerate(ep_mem.retrieve(query, k=2), 1):
    print(f"  [{i}] {ep}")

Storing past conversation episodes...

   [Episodic] Stored episode #1: Session on 2024-01-10: User asked about Python list comprehe...
   [Episodic] Stored episode #2: Session on 2024-01-15: User wanted help with a SQL query joi...
   [Episodic] Stored episode #3: Session on 2024-01-20: User asked about neural network train...
   [Episodic] Stored episode #4: Session on 2024-01-25: User was building a Flask web API. We...
   [Episodic] Stored episode #5: Session on 2024-02-01: User asked about transformer attentio...

────────────────────────────────────────────────────────────

User query: "Can you help me understand how transformers process language?"

Top 2 retrieved episodes:
  [1] Session on 2024-02-01: User asked about transformer attention mechanisms. We walked through self-attention, multi-head attention, and positional encoding.
  [2] Session on 2024-01-10: User asked about Python list comprehensions and lambda functions. We practised filtering lists and mapping transformatio

---
## Step 4 — Long-Term Memory: Remembering Facts About the User

### The Difference from Episodic Memory

Episodic memory recalls *what we talked about*. Long-term memory recalls *facts about the person*.

| Type | What It Stores | Example |
|------|----------------|--------|
| Episodic | Session summaries | "Last week we debugged a Python script" |
| Long-Term | Facts / preferences | "User's name is Alice. She prefers Python. She hates verbose explanations." |

### How We Extract Facts

We use Claude itself as a **fact extractor**. After each conversation, we pass the transcript to Claude and ask it: *"What facts about the user can you extract from this?"*

Claude returns a list like:
```
- User's name is Alice
- User is studying machine learning at university
- User prefers concise explanations with code examples
- User dislikes overly mathematical proofs
```

Each fact is embedded and stored in FAISS. On every new turn, we recall the most relevant facts and inject them into the system prompt.

In [7]:
class LongTermMemory:
    """
    Stores individual facts about the user as vector embeddings.
    Facts are extracted from conversations by Claude and persist indefinitely.
    """

    def __init__(self, embed_dim: int = EMBED_DIM, similarity_threshold: float = 0.70):
        self.index = faiss.IndexFlatIP(embed_dim)
        self.facts: list[str] = []           # parallel list: facts[i] ↔ index vector i
        self.threshold = similarity_threshold  # minimum score to return a fact

    # ── storing facts ────────────────────────────────────────────────────────

    def remember(self, fact: str):
        """Embed a single fact and store it."""
        vec = embed(fact).reshape(1, -1)
        self.index.add(vec)
        self.facts.append(fact)

    # ── recalling facts ──────────────────────────────────────────────────────

    def recall(self, query: str, k: int = 5) -> list[str]:
        """
        Return the K facts most relevant to the query,
        filtered by the similarity threshold.
        """
        if self.index.ntotal == 0:
            return []

        k = min(k, self.index.ntotal)
        q_vec = embed(query).reshape(1, -1)
        scores, indices = self.index.search(q_vec, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0 and float(score) >= self.threshold:
                results.append((float(score), self.facts[idx]))

        results.sort(key=lambda x: x[0], reverse=True)
        return [fact for _, fact in results]

    # ── Claude-based fact extraction ─────────────────────────────────────────

    def extract_and_store(self, conversation_text: str):
        """
        Ask Claude to extract user facts from a conversation transcript,
        then store each extracted fact.
        """
        system = (
            "You are a fact-extraction assistant. "
            "Given a conversation transcript, extract concise facts about the USER only. "
            "Output ONLY a JSON array of short fact strings. "
            "Example: [\"User's name is Alice\", \"User prefers Python\"]\n"
            "If there are no clear facts, return an empty array: []"
        )
        response = claude_client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=512,
            system=system,
            messages=[{"role": "user", "content": conversation_text}],
        )

        raw = response.content[0].text.strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        try:
            facts = json.loads(raw)
        except json.JSONDecodeError:
            print(f"   [LTM] Warning: could not parse facts JSON: {raw[:100]}")
            return

        print(f"   [LTM] Extracted {len(facts)} fact(s) from conversation:")
        for fact in facts:
            print(f"         • {fact}")
            self.remember(fact)

    def __len__(self):
        return self.index.ntotal


print("✅ LongTermMemory class defined.")

✅ LongTermMemory class defined.


### Demo: Extract Facts and Recall Them

Let's feed a sample conversation to the fact extractor, then query the long-term store.

In [8]:
lt_mem = LongTermMemory(similarity_threshold=0.65)

# Simulate a past conversation transcript
sample_transcript = """
User: Hi! I'm Alice, a second-year grad student studying NLP at IIT Bombay.
Assistant: Welcome, Alice! What are you working on?
User: I'm writing my thesis on low-resource language models. I mostly code in Python.
Assistant: That sounds fascinating. Do you have a preferred framework?
User: I love PyTorch. I find TensorFlow confusing. I also hate it when explanations are too verbose.
Assistant: Noted — I'll keep things concise. Any particular aspect of low-resource NLP you want to explore?
User: Yes, I'm especially interested in cross-lingual transfer learning and Hindi-English code switching.
"""

print("Extracting facts from transcript...\n")
lt_mem.extract_and_store(sample_transcript)

print(f"\nTotal facts stored: {len(lt_mem)}")

print(f"\n{'─'*60}")
query = "What programming languages does the user know?"
print(f"\nQuery: \"{query}\"")
recalled = lt_mem.recall(query, k=3)
print("Recalled facts:")
for f in recalled:
    print(f"  • {f}")

Extracting facts from transcript...

   [LTM] Extracted 10 fact(s) from conversation:
         • User's name is Alice
         • User is a second-year grad student
         • User studies NLP at IIT Bombay
         • User is writing a thesis on low-resource language models
         • User codes primarily in Python
         • User prefers PyTorch over TensorFlow
         • User finds TensorFlow confusing
         • User dislikes verbose explanations
         • User is interested in cross-lingual transfer learning
         • User is interested in Hindi-English code switching

Total facts stored: 10

────────────────────────────────────────────────────────────

Query: "What programming languages does the user know?"
Recalled facts:


---
## Step 5 — Putting It All Together: The Memory-Augmented Chat Function

### How the Three Layers Combine

Every time the user sends a message, we do the following before calling Claude:

```
User message: "Can you explain gradient descent?"
                        │
                        ▼
  1. Retrieve from EpisodicMemory  → "Past session: we discussed neural networks"
  2. Recall from LongTermMemory    → "User is Alice, prefers Python, dislikes verbose explanations"
  3. Build memory-augmented system prompt:
       "You are a helpful assistant.
        LONG-TERM FACTS ABOUT THE USER:
          • User's name is Alice
          • User prefers concise explanations
          ...
        RELEVANT PAST CONVERSATIONS:
          • Session on 2024-01-20: discussed neural networks and gradient descent..."
  4. Pass system prompt + ShortTermMemory window to Claude
```

Claude then responds with *personal context* already baked in — without the user needing to repeat anything.

In [9]:
class MemoryAugmentedAssistant:
    """
    A chatbot that combines all three memory layers:
      - ShortTermMemory  : the current conversation window
      - EpisodicMemory   : relevant past conversation summaries
      - LongTermMemory   : persistent facts about the user
    """

    BASE_SYSTEM = (
        "You are a thoughtful, helpful AI assistant. "
        "You have access to memory about the user and past conversations. "
        "Use this context to give personalised, relevant responses. "
        "Do not make up facts — only use what is provided."
    )

    def __init__(
        self,
        stm_max_tokens: int = 3000,
        episodic_k: int = 2,
        ltm_k: int = 5,
    ):
        self.stm   = ShortTermMemory(max_tokens=stm_max_tokens)
        self.ep    = EpisodicMemory()
        self.ltm   = LongTermMemory()
        self.episodic_k = episodic_k
        self.ltm_k      = ltm_k

    # ── memory injection ─────────────────────────────────────────────────────

    def _build_system_prompt(self, user_message: str) -> str:
        """
        Build a system prompt augmented with recalled long-term facts
        and relevant episodic memories.
        """
        sections = [self.BASE_SYSTEM]

        # ── Long-Term Memory ──
        lt_facts = self.ltm.recall(user_message, k=self.ltm_k)
        if lt_facts:
            facts_text = "\n".join(f"  • {f}" for f in lt_facts)
            sections.append(
                f"\n### LONG-TERM FACTS ABOUT THE USER\n{facts_text}"
            )

        # ── Episodic Memory ──
        episodes = self.ep.retrieve(user_message, k=self.episodic_k)
        if episodes:
            ep_text = "\n".join(f"  [{i+1}] {e}" for i, e in enumerate(episodes))
            sections.append(
                f"\n### RELEVANT PAST CONVERSATIONS\n{ep_text}"
            )

        return "\n".join(sections)

    # ── main chat method ─────────────────────────────────────────────────────

    def chat(self, user_message: str, verbose: bool = False) -> str:
        """
        Process a user message and return Claude's response.
        Automatically manages all three memory layers.
        """
        # 1. Add user message to short-term memory
        self.stm.add("user", user_message)

        # 2. Build memory-augmented system prompt
        system_prompt = self._build_system_prompt(user_message)

        if verbose:
            print(f"\n{'═'*60}")
            print("[System Prompt Sent to Claude]")
            print(system_prompt)
            print(f"{'═'*60}")
            print(f"[Short-Term Window] {len(self.stm.get_messages())} messages")

        # 3. Call Claude with STM window + augmented system prompt
        response = claude_client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=1024,
            system=system_prompt,
            messages=self.stm.get_messages(),
        )

        reply = response.content[0].text

        # 4. Add Claude's response to short-term memory
        self.stm.add("assistant", reply)

        return reply

    # ── session management ───────────────────────────────────────────────────

    def end_session(self, session_summary: str):
        """
        Call at the end of a session to:
          1. Store the session summary in episodic memory
          2. Extract and store facts in long-term memory
          3. Clear short-term memory for the next session
        """
        print("\n[Session Ended — Updating Memory]")
        self.ep.store_episode(session_summary)

        # Build full transcript for fact extraction
        transcript = "\n".join(
            f"{m['role'].capitalize()}: {m['content']}"
            for m in self.stm.get_messages()
        )
        self.ltm.extract_and_store(transcript)

        # Clear STM for next session
        self.stm.messages.clear()
        print(f"\nMemory status after session:")
        print(f"  Episodes stored : {len(self.ep)}")
        print(f"  LTM facts stored: {len(self.ltm)}")
        print(f"  STM cleared     : {len(self.stm.get_messages())} messages")


print("✅ MemoryAugmentedAssistant class defined.")

✅ MemoryAugmentedAssistant class defined.


---
## Step 6 — Full Demo: A Memory-Augmented Conversation

Let's run a realistic multi-session demo:

1. **Session A** — We tell the assistant facts about ourselves and ask some questions.
2. **End Session A** — Facts are extracted and the session is summarised into episodic memory.
3. **Session B** — We start fresh (STM cleared), but the assistant still knows who we are!

In [10]:
# ── Create the assistant ─────────────────────────────────────────────────────
assistant = MemoryAugmentedAssistant(stm_max_tokens=3000, episodic_k=2, ltm_k=5)

# Pre-populate with some historical long-term facts (from a previous week)
pre_existing_facts = [
    "User's name is Priya.",
    "Priya is a data science engineer at a fintech startup in Bangalore.",
    "Priya is learning about LLM-based agents and autonomous AI systems.",
    "Priya prefers Python over R, and uses Pandas and PyTorch regularly.",
    "Priya dislikes jargon-heavy explanations without concrete examples.",
]
print("Pre-loading long-term memory with existing facts...")
for f in pre_existing_facts:
    assistant.ltm.remember(f)
    print(f"  ✓ {f}")

print(f"\n{'─'*60}")
print("SESSION A — Starting conversation\n")

# Turn 1
q1 = "Can you explain what a vector database is and why it's useful?"
print(f"Priya: {q1}")
r1 = assistant.chat(q1)
print(f"\nAssistant: {r1}")

Pre-loading long-term memory with existing facts...
  ✓ User's name is Priya.
  ✓ Priya is a data science engineer at a fintech startup in Bangalore.
  ✓ Priya is learning about LLM-based agents and autonomous AI systems.
  ✓ Priya prefers Python over R, and uses Pandas and PyTorch regularly.
  ✓ Priya dislikes jargon-heavy explanations without concrete examples.

────────────────────────────────────────────────────────────
SESSION A — Starting conversation

Priya: Can you explain what a vector database is and why it's useful?

Assistant: ## Vector Databases

A vector database is a specialized database designed to store, index, and query **high-dimensional vectors** (arrays of numbers) efficiently.

### What's a Vector?

When AI models process data (text, images, audio), they convert it into numerical representations called **embeddings** — essentially lists of hundreds or thousands of numbers that capture *meaning and relationships*.

For example, the words "dog" and "puppy" would hav

In [11]:
# Turn 2
q2 = "Great! Now how would I use FAISS in Python to store and search these embeddings?"
print(f"Priya: {q2}")
r2 = assistant.chat(q2)
print(f"\nAssistant: {r2}")

Priya: Great! Now how would I use FAISS in Python to store and search these embeddings?

Assistant: ## Using FAISS in Python

FAISS (Facebook AI Similarity Search) is a popular library for efficient vector similarity search.

### Installation

```bash
pip install faiss-cpu  # CPU version
pip install faiss-gpu  # GPU version (if you have CUDA)
pip install sentence-transformers  # For generating embeddings
```

### Basic Example

```python
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load a model to generate embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample documents
documents = [
    "The cat sat on the mat",
    "Dogs are loyal companions",
    "Python is a great programming language",
    "Machine learning is transforming industries",
    "I love hiking in the mountains",
]

# Generate embeddings
embeddings = model.encode(documents)
print(f"Embedding shape: {embeddings.shape}")  # (5, 384)

dimension = embeddings.shape[1] 

In [12]:
# ── End Session A ────────────────────────────────────────────────────────────
session_a_summary = (
    "Session A (2024-02-10): Priya asked about vector databases and FAISS. "
    "We covered what FAISS is, why vector DBs are useful for semantic search, "
    "and looked at a Python code example for IndexFlatL2."
)

assistant.end_session(session_a_summary)


[Session Ended — Updating Memory]
   [Episodic] Stored episode #1: Session A (2024-02-10): Priya asked about vector databases a...
   [LTM] Warning: could not parse facts JSON: size)

```python
quantizer = faiss.IndexFlatL2(dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, 

Memory status after session:
  Episodes stored : 1
  LTM facts stored: 5
  STM cleared     : 0 messages


In [13]:
print(f"{'─'*60}")
print("SESSION B — Starting a brand-new session (STM is empty)")
print("Watch how the assistant still remembers Priya and past topics.\n")

# Turn 1 in new session — notice the assistant greets by name
q3 = "Hi! I'm trying to build an AI agent that remembers things. Where should I start?"
print(f"Priya: {q3}")
r3 = assistant.chat(q3, verbose=True)   # verbose=True shows what's in the system prompt
print(f"\nAssistant: {r3}")

────────────────────────────────────────────────────────────
SESSION B — Starting a brand-new session (STM is empty)
Watch how the assistant still remembers Priya and past topics.

Priya: Hi! I'm trying to build an AI agent that remembers things. Where should I start?

════════════════════════════════════════════════════════════
[System Prompt Sent to Claude]
You are a thoughtful, helpful AI assistant. You have access to memory about the user and past conversations. Use this context to give personalised, relevant responses. Do not make up facts — only use what is provided.

### RELEVANT PAST CONVERSATIONS
  [1] Session A (2024-02-10): Priya asked about vector databases and FAISS. We covered what FAISS is, why vector DBs are useful for semantic search, and looked at a Python code example for IndexFlatL2.
════════════════════════════════════════════════════════════
[Short-Term Window] 1 messages

Assistant: Hi! Great project — building an agent with memory is a really rewarding challenge

In [14]:
# Turn 2 — follow-up that naturally ties back to the previous session
q4 = "Last time we spoke about FAISS. How does that connect to memory management?"
print(f"Priya: {q4}")
r4 = assistant.chat(q4)
print(f"\nAssistant: {r4}")

Priya: Last time we spoke about FAISS. How does that connect to memory management?

Assistant: Great question! Yes, we did cover FAISS in our last conversation — and it connects very directly to agent memory. Let me tie it together:

---

## FAISS as a Memory Store

Remember the **IndexFlatL2** example we looked at? That same mechanism is essentially what powers long-term memory in an AI agent:

| FAISS Concept | Memory Concept |
|---|---|
| Storing vectors | Saving memories as embeddings |
| Querying nearest neighbours | Retrieving relevant memories |
| Index | Your agent's "memory bank" |

---

## How It Works in Practice

```python
# A memory is just text converted to a vector
memory = "User prefers concise answers"
embedding = embed(memory)          # Convert to vector
index.add(embedding)               # Store in FAISS

# Later, when the agent needs context...
query_vec = embed("How should I respond?")
distances, indices = index.search(query_vec, k=3)
# Returns the 3 most relevant

---
## Step 7 — Visualising the Memory Architecture

Let's print a live status of all three memory layers to see what is stored after our demo.

In [15]:
print("\n" + "═"*65)
print("  MEMORY SYSTEM STATUS")
print("═"*65)

# ── Short-Term Memory ──────────────────────────────────────────────────────
print(f"\n📋 SHORT-TERM MEMORY")
print(assistant.stm.status())
print("  Current window:")
for m in assistant.stm.get_messages():
    preview = m['content'][:80].replace('\n', ' ')
    print(f"    [{m['role']:9s}] {preview}...")

# ── Episodic Memory ────────────────────────────────────────────────────────
print(f"\n🗂️  EPISODIC MEMORY  ({len(assistant.ep)} episode(s) stored)")
for i, ep in enumerate(assistant.ep.episodes, 1):
    print(f"  Episode {i}: {ep[:100]}...")

# ── Long-Term Memory ───────────────────────────────────────────────────────
print(f"\n🧠 LONG-TERM MEMORY  ({len(assistant.ltm)} fact(s) stored)")
for i, f in enumerate(assistant.ltm.facts, 1):
    print(f"  Fact {i:2d}: {f}")

print("\n" + "═"*65)


═════════════════════════════════════════════════════════════════
  MEMORY SYSTEM STATUS
═════════════════════════════════════════════════════════════════

📋 SHORT-TERM MEMORY
ShortTermMemory: 4 messages, 722/3000 tokens used (24.1%)
  Current window:
    [user     ] Hi! I'm trying to build an AI agent that remembers things. Where should I start?...
    [assistant] Hi! Great project — building an agent with memory is a really rewarding challeng...
    [user     ] Last time we spoke about FAISS. How does that connect to memory management?...
    [assistant] Great question! Yes, we did cover FAISS in our last conversation — and it connec...

🗂️  EPISODIC MEMORY  (1 episode(s) stored)
  Episode 1: Session A (2024-02-10): Priya asked about vector databases and FAISS. We covered what FAISS is, why ...

🧠 LONG-TERM MEMORY  (5 fact(s) stored)
  Fact  1: User's name is Priya.
  Fact  2: Priya is a data science engineer at a fintech startup in Bangalore.
  Fact  3: Priya is learning about LLM-

---
## Step 8 — Under the Hood: How FAISS Really Works

Let's take one more look under the hood to solidify the concepts.

### The FAISS Index Types Used Here

| Index | Full Name | Characteristics |
|-------|-----------|----------------|
| `IndexFlatIP` | Flat Inner Product | Exact search, cosine similarity (with normalised vectors), small datasets |
| `IndexFlatL2` | Flat L2 Distance | Exact search, Euclidean distance, slightly different results |
| `IndexIVFFlat` | Inverted File + Flat | Approximate, much faster for millions of vectors |

For production systems with millions of memories, you would switch to `IndexIVFFlat` or `HNSW` (Hierarchical Navigable Small World graphs) for speed.

### Why Normalise Vectors?

```python
# Without normalisation: inner product is affected by vector magnitude
# (a longer sentence would dominate just because it has more words)

# With normalisation: every vector has length 1
# inner_product(a, b) = cos(angle_between_a_and_b)
# → pure semantic similarity, independent of sentence length
```

In [16]:
# ── Demonstrate the FAISS search internals ──────────────────────────────────
print("FAISS search internals demo")
print("─" * 50)

# Build a small toy index
toy_sentences = [
    "I love eating pizza and pasta.",
    "Machine learning is a subfield of AI.",
    "The Eiffel Tower is in Paris.",
    "Deep learning uses neural networks.",
    "Italy is famous for its cuisine.",
]

toy_index = faiss.IndexFlatIP(EMBED_DIM)
print("Embedding and indexing sentences...")
for i, s in enumerate(toy_sentences):
    vec = embed(s).reshape(1, -1)
    toy_index.add(vec)
    print(f"  [{i}] {s}")

print(f"\nIndex size: {toy_index.ntotal} vectors of dimension {toy_index.d}")

# Search
query = "What's a good Italian dish?"
q_vec = embed(query).reshape(1, -1)
scores, indices = toy_index.search(q_vec, k=3)

print(f"\nQuery: \"{query}\"")
print("Top 3 results (cosine similarity):")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    print(f"  #{rank} (score={score:.4f}): {toy_sentences[idx]}")

FAISS search internals demo
──────────────────────────────────────────────────
Embedding and indexing sentences...
  [0] I love eating pizza and pasta.
  [1] Machine learning is a subfield of AI.
  [2] The Eiffel Tower is in Paris.
  [3] Deep learning uses neural networks.
  [4] Italy is famous for its cuisine.

Index size: 5 vectors of dimension 1536

Query: "What's a good Italian dish?"
Top 3 results (cosine similarity):
  #1 (score=0.5730): Italy is famous for its cuisine.
  #2 (score=0.4916): I love eating pizza and pasta.
  #3 (score=0.0797): Deep learning uses neural networks.


---
## Summary

Congratulations! You have built a complete three-layer memory system for an AI agent. Here is what we covered:

```
┌─────────────────────────────────────────────────────────────────────┐
│                     What You Built Today                            │
│                                                                     │
│  ShortTermMemory                                                    │
│  ├── Token-aware sliding window using tiktoken                      │
│  ├── Drops oldest messages when context limit exceeded              │
│  └── Always passed directly to the LLM                              │
│                                                                     │
│  EpisodicMemory                                                     │
│  ├── Session summaries stored as FAISS vectors (IndexFlatIP)        │
│  ├── Normalised embeddings → cosine similarity search               │
│  └── Top-K relevant episodes injected into system prompt            │
│                                                                     │
│  LongTermMemory                                                     │
│  ├── Facts extracted from conversations by Claude (LLM-as-extractor)│
│  ├── Each fact stored as a FAISS vector                             │
│  └── Semantically recalled and injected into every system prompt    │
│                                                                     │
│  MemoryAugmentedAssistant                                           │
│  ├── chat()       — combines all three layers on every turn         │
│  └── end_session()— extracts facts, stores episode, clears STM      │
└─────────────────────────────────────────────────────────────────────┘
```

### Key Concepts Learned

| Concept | What It Does |
|---------|-------------|
| **Embeddings** | Turn text into vectors capturing semantic meaning |
| **FAISS IndexFlatIP** | Exact cosine similarity search over stored vectors |
| **Vector normalisation** | Makes inner product equal to cosine similarity |
| **tiktoken** | Counts tokens to enforce context window limits |
| **Sliding window (STM)** | Keeps recent messages; drops oldest on overflow |
| **Episodic retrieval** | Semantic search over session summaries |
| **Fact extraction** | Using the LLM itself to extract structured knowledge |
| **System prompt injection** | Inserting retrieved context before every LLM call |

---

### Try It Yourself

1. **Increase the STM limit** and observe that fewer messages are dropped. Try setting it to 100 tokens and watch more aggressive truncation.
2. **Store 10+ episodes** and compare retrieval quality when asking about different topics. Does FAISS find the right ones?
3. **Modify the fact extractor prompt** to also capture the user's emotional state or goals. Run a conversation and see what facts get stored.
4. **Swap `IndexFlatIP` for `IndexFlatL2`** in EpisodicMemory. How do the results differ? (Hint: L2 minimises distance while IP maximises similarity — the ranking may change.)
5. **Production scaling**: Replace `IndexFlatIP` with `faiss.IndexIVFFlat(quantizer, EMBED_DIM, nlist=10)` and observe that you need to call `index.train(vectors)` before adding items — this is how FAISS handles millions of vectors efficiently.